# Notebook 7 — Dati visuales web

**Curso:** EITC/AI/GVAPI — European IT Certification, Google Vision API
**Lezione 7:** Understanding Web Visual Data (Detecting Web Entities and Pages)

In [ ]:
import os
# os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "ruta/a/tu/clave.json"

from google.cloud import vision
client = vision.ImageAnnotatorClient()
print("Cliente listo.")


## Detección web (`web_detection`)

`web_detection` busca tu immagine en el índice web y el grafo de conocimiento de Google, y te dice si existe en algún lugar de internet, con qué páginas se relaciona y qué entidades representa.

**Los seis elementos que te devuelve la risposta:**

1. **`web_entities`**: oggetti, monumentos, logotipos u otros elementos visuales relacionados, cada uno con `description` y `score`
2. **`full_matching_images`**: immagini idénticas o casi idénticas encontradas en internet
3. Imágenes parcialmente coincidentes: comparten elementos visuales, pero no son coincidencia exacta
4. **`pages_with_matching_images`**: páginas web que contienen immagini coincidentes
5. **`visually_similar_images`**: immagini similares anche se no coincidan exactamente
6. **`best_guess_labels`**: la etichetta que mejor describe la immagine, según Google

**Un matiz a tener en cuenta**: no todos los enlaces que te devuelve la API siguen activos — algunos pueden dar errore 404 (página no encontrada), así que verifica siempre prima di dar por buena una coincidencia.

In [ ]:
def web_detect(image_path):
    with open(image_path, "rb") as image_file:
        content = image_file.read()

    image = vision.Image(content=content)
    response = client.web_detection(image=image)
    annotations = response.web_detection

    print("Mejor etiqueta estimada:", [label.label for label in annotations.best_guess_labels])

    for entity in annotations.web_entities:
        print("Entidad:", entity.description, "- score:", entity.score)

    return annotations


# web_detect("mi_foto.jpg")


## Ejercicio 1 — Comprobar enlaces válidos frente a rotos

Completa `verificar_enlaces` para que, dada una immagine, recorra `full_matching_images` y `pages_with_matching_images`, haga una petición HTTP `HEAD` a cada URL con la librería `requests`, y devuelva dos listas: `validos` (codice de estado 200) y `rotos` (cualquier otro codice o errore de conexión).

Pista: usa `requests.head(URL, timeout=5)` dentro de un `try/except` para capturar errorei de conexión.

In [ ]:
import requests

def verificar_enlaces(image_path):
    annotations = web_detect(image_path)
    validos, rotos = [], []

    urls = [m.url for m in annotations.full_matching_images] + [
        p.url for p in annotations.pages_with_matching_images
    ]

    for url in urls:
        # TODO: haz una petición HEAD y clasifica la url en "validos" o "rotos"
        try:
            ...
        except requests.RequestException:
            rotos.append(url)

    return validos, rotos


# validos, rotos = verificar_enlaces("mi_foto.jpg")
# print("Válidos:", len(validos), "- Rotos:", len(rotos))


<details>
<summary><strong>Ver soluzione</strong></summary>

```Python
import requests

def verificar_enlaces(image_path):
    annotations = web_detect(image_path)
    validos, rotos = [], []

    URLs = [m.URL for m in annotations.full_matching_images] + [
        p.URL for p in annotations.pages_with_matching_images
    ]

    for URL in URLs:
        try:
            risposta = requests.head(URL, timeout=5)
            if risposta.status_code == 200:
                validos.append(URL)
            else:
                rotos.append(URL)
        except requests.RequestException:
            rotos.append(URL)

    return validos, rotos


validos, rotos = verificar_enlaces("mi_foto.jpg")
print("Válidos:", len(validos), "- Rotos:", len(rotos))
```

</details>

## Ejercicio 2 — Generador automático de etichette ("auto-tagging")

Completa `generar_etichette_automaticas` para que devuelva las tres `description` de `web_entities` con mayor `score`, simulando el caso de uso de generación automática de etichette para una plataforma de fotos (descrito en el material original con el esempio de Flickr).

In [ ]:
def generar_etiquetas_automaticas(image_path, top_n=3):
    annotations = web_detect(image_path)

    # TODO: ordena annotations.web_entities por score descendente y toma las top_n descriptions
    entidades_ordenadas = ...
    return [e.description for e in entidades_ordenadas[:top_n]]


# print(generar_etiquetas_automaticas("mi_foto.jpg"))


<details>
<summary><strong>Ver soluzione</strong></summary>

```Python
def generar_etichette_automaticas(image_path, top_n=3):
    annotations = web_detect(image_path)
    entidades_ordenadas = sorted(annotations.web_entities, key=lambda e: e.score, reverse=True)
    return [e.description for e in entidades_ordenadas[:top_n]]


print(generar_etichette_automaticas("mi_foto.jpg"))
```

</details>

## Ejercicio 3 — Detector de reutilización de immagini

Completa `rilevare_reutilizacion` para que compare los risultati de `full_matching_images` (coincidencias exactas) y `visually_similar_images` (similares pero no exactas) entre dos immagini — una propia y otra descargada de internet — y devuelva un resumen con el número de coincidencias de cada tipo y la lista de páginas donde aparece cada una.

In [ ]:
def detectar_reutilizacion(image_path):
    annotations = web_detect(image_path)

    # TODO: construye el resumen con:
    # "coincidencias_exactas" (num. de full_matching_images)
    # "similares" (num. de visually_similar_images)
    # "paginas" (lista de urls de pages_with_matching_images)
    resumen = ...
    return resumen


# print(detectar_reutilizacion("mi_foto_compartida.jpg"))


<details>
<summary><strong>Ver soluzione</strong></summary>

```Python
def rilevare_reutilizacion(image_path):
    annotations = web_detect(image_path)

    resumen = {
        "coincidencias_exactas": len(annotations.full_matching_images),
        "similares": len(annotations.visually_similar_images),
        "paginas": [p.URL for p in annotations.pages_with_matching_images],
    }
    return resumen


print(rilevare_reutilizacion("mi_foto_compartida.jpg"))
```

</details>

---

**Siguiente paso:** abre `08_Forme_y_Oggetti_Pillow.ipynb` para aprender a dibujar sobre las immagini los risultati que has ido obteniendo en los notebooks anteriores.